# From One Neuron to a Network

You've already built a neuron. Logistic regression (Chapter 4) takes a weighted sum of the features, adds a constant, and squashes the result into a probability. Neural networks are built from exactly this unit. This notebook starts from one unit, finds what it can and can't do, and ends with a small network that does what one unit can't (ISL §10.1–10.2).

**Where this sits in modern AI.** Every large language model is built from layers like the ones at the end of this notebook: multiply by a matrix of weights, add a bias, apply an activation, and repeat. GPT-3 has 175 billion weights. Today we set 9 numbers by hand, and see why the rest can't be set that way.

### By the end, you'll be able to

1. Write logistic regression as a neuron: a weighted sum $z = \mathbf{w}\cdot\mathbf{x} + b$ followed by an activation, and translate between $\beta$ and $\mathbf{w}, b$.
2. Read one neuron's decision boundary as a line in feature space, and explain why it is a line whatever the activation.
3. Explain why no single neuron can classify XOR.
4. Write a layer of weighted sums as $W\mathbf{x} + \mathbf{b}$, and explain why weighted sums alone can't make classes separable: only the activation bends space.
5. Explain how combining neurons changes the boundary: each hidden unit adds a cliff along its own line, the output neuron adds the cliffs up, and its boundary can be built from pieces of several lines.
6. Explain the output unit as one neuron drawing one straight line in h-space, the space of the hidden units' outputs.
7. Predict which region an output neuron selects, given its hidden units' lines and its weights.

> **How to read the code in this notebook.** There are two kinds of code cells.
>
> - **Code worth reading.** Cells that use standard libraries, such as numpy (`np.`), do everyday tasks you'll do yourself. Each is introduced with **Code worth reading** in bold. Read these line by line.
> - **Demo cells.** Cells that import from `ch_6_and_10_demos_helpers` draw a plot or build an interactive demo. That file was written to keep the notebook on ideas, so you never need to open it.

In [ ]:
import numpy as np

<a id="section-1"></a>

## 1 · You've already built a neuron

In Chapter 4, logistic regression set the log-odds equal to a linear combination of the features. With two features:

$$
\ln\!\left(\frac{p}{1-p}\right) = z = \beta_0 + \beta_1 X_1 + \beta_2 X_2
$$

Exponentiating both sides and solving for $p$ gave the logistic function:

 $$p = \dfrac{1}{1 + e^{-z}}$$

Neural networks call this the **sigmoid** function.

Neural networks represent the same model using vector notation. 

- Put the feature values in a vector $\mathbf{x} = (x_1, x_2)$, 
- Instead of $\beta$ , call the coefficients **weights** and represent them as a **weight vector** $\mathbf{w} = (w_1, w_2)$, 
- Call the intercept the **bias**, $b$. 

The score is then a dot product plus the bias:

$$
z = \mathbf{w}\cdot\mathbf{x} + b = w_1 x_1 + w_2 x_2 + b
$$

The output of a **neuron** is $g(z)$, where $g$ is called the **activation function**. 

We'll use three choices for $g$. Each one reports the same score $z$ in a different way:

- **Sigmoid**, $g(z) = \dfrac{1}{1 + e^{-z}}$: a probability between 0 and 1. With the sigmoid, one neuron is exactly logistic regression.
- **Step:** 1 if $z > 0$, and 0 otherwise. It reports only which side of 0 the score is on: a predicted class instead of a probability. The earliest artificial neurons, called **perceptrons**, used the step.
- **ReLU** ("rectified linear unit"), $g(z) = \max(0, z)$: 0 when $z \le 0$, and $z$ itself when $z > 0$. It's the most common choice inside modern networks.

In [ ]:
from ch_6_and_10_demos_helpers import plot_activations

plot_activations();

The whole unit, called a **neuron** or **unit**, is:

$$
\text{output} = g(z) = g(\mathbf{w}\cdot\mathbf{x} + b)
$$

With the sigmoid, the output is the probability $p$.

In [ ]:
from ch_6_and_10_demos_helpers import plot_neuron_diagram

plot_neuron_diagram();

| Chapter 4 | Neural networks |
| --- | --- |
| coefficients $\beta_1, \dots, \beta_p$ | weights $w_1, \dots, w_p$ |
| intercept $\beta_0$ | bias $b$ |
| evidence score (log-odds) | score $z$ |
| logistic function | sigmoid activation |

<details>
<summary>Why is it called a neuron?</summary>

The name comes from a loose analogy with a brain cell. A biological neuron receives signals from other cells, and some connections count for more than others (the weights). If the combined input is strong enough, the cell fires a signal of its own (the activation).

The analogy stops there. Real neurons fire in spikes over time, do part of their computation in their branches, and don't learn by anything like gradient descent. Treat "neuron" as a name, not a description of the brain: everything in this notebook is a weighted sum and a function.

</details>

**Ask:** A fitted logistic regression has log-odds $-3 + 2X_1 - 0.5X_2$. As a neuron, what are $\mathbf{w}$ and $b$? What does it output for $\mathbf{x} = (1, 2)$?

<details>
<summary>Answer</summary>

$\mathbf{w} = (2, -0.5)$ and $b = -3$. The score is $z = 2(1) - 0.5(2) - 3 = -2$, so the output is $p = \dfrac{1}{1 + e^{2}} \approx 0.12$.

</details>

<a id="section-2"></a>

## 2 · One neuron draws one line

**The setup.** We have a classification problem with two features, $x_1$ and $x_2$, and two classes. Each data point is a point $\mathbf{x} = (x_1, x_2)$ in a plane whose two axes are the **features**. This plane is called **feature space**. The neuron's job is to predict each point's class. (In every plot of data in this notebook, **blue circles are class 1** and **orange triangles are class 0**, as in earlier chapters.) Geometric interpretation of class-prediction: draw a decision boundary through feature space, with one class mostly on one side and the other class mostly on the other.

<details>
<summary>A concrete example: the Default data</summary>

In Chapter 4's `Default` data, each customer has two features, credit card **balance** ($x_1$) and **income** ($x_2$), and one of two classes: **defaulted** ($y=1$) or **didn't default** ($y=0$).

A customer's balance of \$1,500 and income of \$40,000 is the single point $\mathbf{x} = (1500,\ 40000)$ in feature space. The plane's horizontal axis is balance, its vertical axis is income, and the 10,000 customers are 10,000 points as (balance, income) pairs. Here, the line would have the customers who defaulted mostly on one side, and those who didn't mostly on the other.

</details>

**Careful: this is a different plane from the ridge notebook's pictures.** There, the axes were the *coefficients* $\beta_1$ and $\beta_2$, and every point in the plane was a whole model (the RSS ovals and the budget regions lived there). In this notebook, the axes are the *features*. Each dot is one data point, and a model shows up as its decision boundary, a whole line. Here is the same model drawn in both planes:

In [ ]:
from ch_6_and_10_demos_helpers import plot_feature_vs_weight_space

plot_feature_vs_weight_space();

- **Left, feature space:** 40 data points, and one model, $\mathbf{w} = (1, 1)$ with $b = -1$, shown by its decision boundary, the line $x_1 + x_2 - 1 = 0$. (The model itself is the whole rule that scores and labels every point; the line is where its prediction switches.) Almost every picture in this notebook is in this plane.
- **Right, weight space:** the same model as a single dot at $\mathbf{w} = (1, 1)$, with $b$ held at $-1$, since only two axes fit. (Weight space, also called *parameter space*, really has one axis for every parameter, weights and biases alike.) This is the kind of plane ridge's pictures used.

**The step activation.** To keep the pictures crisp, most of this notebook uses the step from Section 1. A step neuron computes the score $z = \mathbf{w}\cdot\mathbf{x} + b$ and predicts

$$
\text{prediction} = \begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{otherwise} \end{cases}
$$

In [ ]:
from ch_6_and_10_demos_helpers import plot_step_function

plot_step_function();

**Ask:** Modern neural networks are trained with gradient descent. Could they use the step activation?

<details>
<summary>Hint</summary>

Gradient descent changes each weight by an amount proportional to the slope of the loss with respect to that weight.

1. Take a point whose score is $z = 0.8$, so the step outputs 1. Nudge one weight slightly, so the score becomes $0.81$. What happens to the step's output?
2. What happens to the loss, then? What slope does gradient descent see for that weight?
3. Look at the plot above. Is there anywhere the answer to question 1 would be different?

</details>

<details>
<summary>Answer</summary>

Not usefully.

- **The output doesn't move.** Nudging a weight changes the score $z$ a little, but unless $z$ crosses 0, the step's output doesn't change at all: 1 stays 1, and 0 stays 0. So the loss doesn't change either, and its slope with respect to that weight is 0.
- **That's true almost everywhere.** The step is flat everywhere except at $z = 0$, where it jumps. At the jump, the slope isn't defined at all.
- **So gradient descent never moves.** Its update, new value $=$ old value $- \eta \times$ slope, multiplies $\eta$ by 0. Even a badly misclassified point gives no hint about which way to turn $\mathbf{w}$.

That's why modern networks use activations whose slopes carry information. The sigmoid changes a little whenever $z$ changes a little, so its slope is never 0. ReLU has slope 1 wherever $z > 0$. (The original perceptron wasn't trained by gradient descent. It used its own mistake-driven update rule.)

The step still earns its place in this notebook as a thinking tool. Its outputs are exactly 0 or 1, so the arithmetic in Section 4 is easy to check by hand.

</details>

**Why one neuron's boundary is a line:** Take the example model, $\mathbf{w} = (1, 1)$ and $b = -1$, so the score is $z = x_1 + x_2 - 1$. Picture feature space as a "floor", with various options for the vertical axis: 

1. **$z$:** a tilted plane. 
2. **$p = g(z)$, the sigmoid:** an s-shaped sheet. 
3. **The perceptron's step prediction:** the plane becomes a cliff, and its edge runs along the same line.

The small diagram above each panel shows which part of the neuron that panel pictures. Everything else is faded:

- **The weighted sum $z$:** the incoming edges and the unit's outline are highlighted.
- **The activation $g(z)$:** the unit's fill and its outgoing arrow are highlighted.

Section 4 uses the same small diagrams for a bigger network.

In [ ]:
from ch_6_and_10_demos_helpers import plot_score_surfaces

plot_score_surfaces();

In all three panels, walking perpendicular to $\mathbf{w}$ (parallel to the black line) never changes the height. Each surface is one profile (a slope, an S, or a step) slid sideways along a straight track, so every level of it is a straight line. **The activation decides the shape of the profile, and the score $z$ alone decides where the boundary is.** That's also why logistic regression's boundary is a line.

The direction of $\mathbf{w}$ matters too: $\mathbf{w}$ points **perpendicular to the boundary**, toward the side predicted 1, and it's the direction in which the score rises fastest. So when we say a unit "points in the direction $(1, 1)$," we mean its $\mathbf{w}$ points that way, and its boundary runs across it.

So **one perceptron draws one line in feature space**, and predicts 1 on one side and 0 on the other. (With $p$ features, feature space has $p$ dimensions, and the boundary is a flat surface called a *hyperplane*. We'll stay with two features so we can see it.)

<a id="section-3"></a>

## 3 · What one neuron can't do: XOR

A single sigmoid neuron is logistic regression, so it can do whatever logistic regression can: separate two classes that a line can separate. Section 2 showed why its boundary is always a line. So what *can't* it do?

Here is a small, famous example. There are 100 points in four clusters, at the corners of a square:

- **Blue (1):** the corners $(1, 0)$ and $(0, 1)$, where exactly one of $x_1, x_2$ is "on."
- **Orange (0):** the corners $(0, 0)$ and $(1, 1)$, where both are off or both are on.

This is the **XOR** ("exclusive or") pattern from logic: true when exactly one input is true.

<a id="demo-p"></a>

### Demo P: Try to separate XOR with one neuron

The demo shows one step neuron over the XOR data. Its three sliders control the line:

- **direction of w** turns the line (and $\mathbf{w}$, which is perpendicular to it).
- **line's distance from origin** slides the line along $\mathbf{w}$. The demo sets $b$ for you, as $b = -\lVert\mathbf{w}\rVert \times \text{distance}$, and shows it in the readout.
- **length of w** stretches $\mathbf{w}$. For a step neuron it changes no prediction, because doubling $\mathbf{w}$ and $b$ doubles every score without changing its sign.

The left panel shows the two sides of the line, with mistakes circled. The right panel plots each point's prediction against its signed distance from the line.

In [ ]:
from ch_6_and_10_demos_helpers import neuron_demo, xor_data

X_xor, y_xor = xor_data()
neuron_demo(X_xor, y_xor, activation='step', size_percent=80, start=(0, 1, 0.5))

**Ask:** Predict first: can any setting make zero mistakes? Then try. What is the fewest you can get?

<details>
<summary>Hint</summary>

Draw the segment from $(1, 0)$ to $(0, 1)$, and the segment from $(0, 0)$ to $(1, 1)$. Where do they meet? Which side of your line would that point have to be on?

</details>

<details>
<summary>Answer</summary>

The fewest is 25 mistakes, one whole cluster. For example, direction 45° with distance 0.3 gets three corners right.

No line can do better. The two blue corners sit diagonally across from each other, and so do the two orange corners, so the segment joining the blue corners crosses the segment joining the orange ones at the center of the square. A line that keeps both blue corners on its blue side keeps the whole blue segment there, and a line that keeps both orange corners on its orange side keeps the whole orange segment there. The two segments share a point, which would have to be on both sides at once.

So **no single neuron can classify XOR**, whatever its activation: its boundary is always one line.

</details>

<a id="section-4"></a>

## 4 · Combining neurons

One neuron draws one line. So use two neurons, and feed their outputs into a third.

Every unit works exactly like the single neuron in Section 1: a score $z$ (weights times inputs, plus a bias), then the activation $g(z)$. The units are arranged in **layers**, and a superscript in parentheses says which layer a symbol belongs to. (It's a label, not an exponent.)

- **Layer 1, the hidden layer,** has two units that read the features. Unit $k$ has score $z^{(1)}_k = w^{(1)}_{k1}x_1 + w^{(1)}_{k2}x_2 + b^{(1)}_k$, so each draws its own line. Call their outputs $h_1 = g(z^{(1)}_1)$ and $h_2 = g(z^{(1)}_2)$. A unit's output is also called its **activation**: $g$ is the activation *function*, and $h_k$ is the activation it produces.
- **Layer 2, the output layer,** has one unit that reads $h_1$ and $h_2$, not the features: $z^{(2)} = w^{(2)}_1 h_1 + w^{(2)}_2 h_2 + b^{(2)}$. Its output $g(z^{(2)})$ is the network's prediction, blue when $z^{(2)} > 0$.

In $w^{(\ell)}_{kj}$, the superscript $\ell$ is the layer, $k$ is the unit the weight feeds into, and $j$ is the input it comes from: first the unit, then the input. The output layer has only one unit, so its weights need just one subscript.

The layer-1 units are called **hidden units** because they're neither the inputs nor the output, and $h$ stands for "hidden." The features themselves aren't a layer of units: they're just the numbers fed in. So this network has two layers of units, one hidden and one output.

In [ ]:
from ch_6_and_10_demos_helpers import plot_network_diagram

plot_network_diagram();

**Why would this draw anything but a line?** Go back to Section 2's picture of a score as a height above feature space.

- **One neuron:** its surface changes height only in the direction of $\mathbf{w}$. Walk parallel to its line and the height never changes. That's why every level of the surface, including the boundary, is a straight line.
- **The output unit:** its score $z^{(2)} = w^{(2)}_1 h_1 + w^{(2)}_2 h_2 + b^{(2)}$ adds two such surfaces, each scaled by a weight. Each surface changes height only across *its own* line. Their sum changes height across *both* lines, so it isn't a single profile slid along one track anymore. Its zero level, the network's boundary, doesn't have to be one line.

To see exactly what the sum looks like, use step activations in the hidden layer: $h_k$ is 1 on its line's positive side and 0 on the other. (With 1 and 0 the arithmetic is easy to check.) Then each hidden unit's surface is a **cliff** along its line, and the output score is a sum of cliffs.

Here is a network for XOR. Both hidden units point in the direction $(1, 1)$, with lines at different places along the diagonal:

| Unit | Score | Its line | Output is 1 when... |
| --- | --- | --- | --- |
| hidden unit 1 | $z^{(1)}_1 = x_1 + x_2 - 0.5$ | $x_1 + x_2 = 0.5$ | at least one of $x_1, x_2$ is on (OR) |
| hidden unit 2 | $z^{(1)}_2 = x_1 + x_2 - 1.5$ | $x_1 + x_2 = 1.5$ | both are on (AND) |
| output unit | $z^{(2)} = h_1 - h_2 - 0.5$ | | |

So every $w^{(1)}_{kj} = 1$, $b^{(1)}_1 = -0.5$, $b^{(1)}_2 = -1.5$, $w^{(2)}_1 = 1$, $w^{(2)}_2 = -1$, and $b^{(2)} = -0.5$.

<a id="layer-weighted-sums"></a>

### Aside: what a layer of weighted sums can do

Before following the data through this network, it helps to know what the hidden layer's first operation can and can't do.

Each hidden unit computes its own weighted sum, $z^{(1)}_k = w^{(1)}_{k1}x_1 + w^{(1)}_{k2}x_2 + b^{(1)}_k$. Its sign says which side of that unit's line a point is on, and its size says how far away. So together, the two scores $(z^{(1)}_1, z^{(1)}_2)$ give each point **new coordinates**, measured from the hidden units' lines instead of from the $x_1$ and $x_2$ axes. What does that change of coordinates do to the plane?

<a id="demo-q"></a>

### Demo Q: What a layer does to the plane

The left panel draws a square grid over feature space, with the XOR data. The right panel shows where every grid line and every point goes. Lines of constant $x_1$ are dark gray, and lines of constant $x_2$ are light gray.

- **weights** chooses both hidden units' weights and biases (the readout lists them). The first choice is our XOR network's hidden layer.
- **activation** applies nothing (the weighted sums alone), the step, or the sigmoid to each score.

The readout says whether one straight line could separate the two classes in the new space.

In [ ]:
from ch_6_and_10_demos_helpers import layer_map_demo

layer_map_demo(size_percent=80)

**Ask:** First, with **activation** set to "none," try every choice of **weights**. Do the grid lines ever bend? Does any choice make the classes separable by one line? Then turn on the step, and then the sigmoid. What changes?

<details>
<summary>Hint</summary>

With no activation, watch the grid: what stays true about the lines in every choice? For the classes to become separable, the two blue clusters would have to end up on the same side of some line, with both orange clusters on the other side. Could moving, turning, or stretching the plane do that?

</details>

<details>
<summary>Answer</summary>

**With no activation, the grid never bends, and no choice makes the classes separable.** The weighted sums can shift the plane (the bias), turn and stretch it (two units pointing different ways), slant it, or flatten it onto a line (our XOR layer, where both units point the same way). But straight lines stay straight, parallel lines stay parallel, and nothing folds. So the two blue clusters, which sit in opposite corners, stay separated by the orange ones.

**The activations bend the grid.** The sigmoid curves the grid lines, and the step snaps whole regions onto the corners of a square. With our XOR layer, both activations make the classes separable by one line.

**But an activation alone isn't enough.** With the units pointing different ways, the sigmoid bends the grid and the classes are still interleaved. Where the hidden units' lines are matters as much as the bend. (Practice Question 16 comes back to this.)

</details>

> **What a layer of weighted sums can do.** It can shift, turn, stretch, slant, or flatten the plane, but straight lines stay straight and parallel lines stay parallel. (In linear algebra terms, it's a *linear map*.) So it can't make classes separable that a line couldn't already separate. **Only the activation bends space.**

<a id="forward-pass"></a>

### Following the data through the network

Every layer does the same two things: a **weighted sum**, then an **activation**. So the network computes in four steps, two per layer, and each step turns one space into the next:

- **The hidden layer** (row 1 below): its weighted sums turn each point $(x_1, x_2)$ into two scores $(z^{(1)}_1, z^{(1)}_2)$, and its activations turn those into new coordinates $(h_1, h_2)$ in **h-space**. (h-space is our name for it. In machine learning, a hidden layer's activations are called its **representation** of the data, and the space they live in is often called a **latent space**. Week 10's embeddings are this same idea.)
- **The output unit** (row 2): its weighted sum turns $(h_1, h_2)$ into one number, $z^{(2)}$, on a **number line**, and its activation cuts that line at 0, snapping each point to an output of 0 or 1: the predicted class.

Each panel is a space the data passes through. Each arrow is one step, and the small diagram above it highlights the part of the network that does that step, as in Section 2. h-space appears twice: it's the hidden layer's output and the output unit's input.

In [ ]:
from ch_6_and_10_demos_helpers import plot_forward_strip

plot_forward_strip();

Row 1 is the aside in action. Each hidden unit's score measures where a point sits relative to that unit's line, so the weighted sums give every point new coordinates. That step is linear, so it can't separate anything. Here it flattens the plane onto **one line** in $z^{(1)}$-space, because both hidden units point in the same direction, $(1, 1)$ (the first choice in Demo Q). Along that line the classes are still interleaved: orange, blue, orange. The **activations** do the separating: the switch lines $z^{(1)}_1 = 0$ (solid) and $z^{(1)}_2 = 0$ (dashed) cut the line into three pieces, and each piece snaps onto one corner of h-space.

**Ask:** The middle panel of row 1 is two-dimensional, but the middle panel of row 2 is a one-dimensional number line. Use the small network diagrams above the arrows to explain why.

<details>
<summary>Hint</summary>

Each unit's score is one coordinate. How many units does each layer have?

</details>

<details>
<summary>Answer</summary>

A layer's scores form a vector with **one entry per unit**.

- The hidden layer has **two** units, so $\mathbf{z}^{(1)} = \big(z^{(1)}_1, z^{(1)}_2\big)$ has two coordinates. Row 1's weighted sums map the 2D feature space into a 2D space.
- The output layer has **one** unit, so $z^{(2)}$ is a single number. Row 2's weighted sum maps the 2D h-space down onto a 1D line.

In general, a layer's weighted sums map a space with one dimension per *input* into a space with one dimension per *unit*. So vectors of scores and of activations live in their own spaces, one per layer. A layer with fewer units than inputs maps down into fewer dimensions, as the output layer does here. A layer with *more* units than inputs maps up into more dimensions; we'll see networks like that in a later lesson.

</details>

**Ask:** In row 1, each dashed line in feature space lands on a single ◇ in $z^{(1)}$-space: a whole line collapses to one point. Why? Which lines collapse to single points in row 2?

<details>
<summary>Hint</summary>

Take two data points on the same dashed line, such as $(1, 0)$ and $(0, 1)$, and compute both hidden scores for each. What stays the same along a dashed line?

</details>

<details>
<summary>Answer</summary>

**Row 1.** Along each dashed line, $x_1 + x_2$ has the same value. Both hidden units have weights $(1, 1)$, so both scores depend on a data point only through $x_1 + x_2$: $z^{(1)}_1 = (x_1 + x_2) - 0.5$ and $z^{(1)}_2 = (x_1 + x_2) - 1.5$. Every data point on the line gets the same two scores, so the whole line lands on one point. For example, $(1, 0)$ and $(0, 1)$ both land at $(0.5, -0.5)$. Since every line collapses this way, the whole plane flattens onto the line of ◇s. (In linear algebra terms, the map has rank one: both columns of $W^{(1)}$ are $(1, 1)$, so $W^{(1)}\mathbf{x}$ is always a multiple of $(1, 1)$.)

**Row 2.** The output unit's score, $z^{(2)} = h_1 - h_2 - 0.5$, depends on a point only through $h_1 - h_2$. Along each dashed line in h-space, $h_1 - h_2$ is the same, so the score is the same, and the whole line lands on its own tick of the number line. The line labeled 0 lands at 0, where the activation cuts.

An output layer with one unit always works like this: it measures one direction, so it always flattens its input space onto a line. The hidden layer flattened the plane here only because both of its units happen to measure the same direction.

</details>

<details>
<summary>Why do the points land on a line here?</summary>

This uses the matrix form of a layer, which the [notation section](#notation) near the end of Section 4 introduces properly: the two weighted sums together are $\mathbf{z}^{(1)} = W^{(1)}\mathbf{x} + \mathbf{b}^{(1)}$, where row $k$ of $W^{(1)}$ holds hidden unit $k$'s weights.

**A matrix times a vector is a combination of the matrix's columns.** Multiply out a $2 \times 2$ matrix times a vector, and group the terms by $x_1$ and $x_2$:

$$
\begin{bmatrix} a & b \\ c & d \end{bmatrix}\begin{bmatrix} x_1 \\ x_2 \end{bmatrix}
= \begin{bmatrix} a x_1 + b x_2 \\ c x_1 + d x_2 \end{bmatrix}
= x_1 \begin{bmatrix} a \\ c \end{bmatrix} + x_2 \begin{bmatrix} b \\ d \end{bmatrix}
$$

So $W\mathbf{x}$ is always "$x_1$ times the first column, plus $x_2$ times the second column." Whatever $\mathbf{x}$ is, the result is some combination of the columns. The set of all such combinations is called the **column space** of $W$: it's every place $W\mathbf{x}$ can possibly land.

- If the two columns point in **different directions**, their combinations reach every point of the plane. The column space is the whole plane.
- If the two columns point in the **same direction** (one is a multiple of the other), every combination is just a longer or shorter copy of that one direction. The column space is a single **line through the origin**.

**Our hidden layer.** Here

$$
W^{(1)} = \begin{bmatrix} 1 & 1 \\ 1 & 1 \end{bmatrix},
$$

and both columns are $(1, 1)$. So

$$
W^{(1)}\mathbf{x} = x_1\begin{bmatrix} 1 \\ 1 \end{bmatrix} + x_2\begin{bmatrix} 1 \\ 1 \end{bmatrix} = (x_1 + x_2)\begin{bmatrix} 1 \\ 1 \end{bmatrix}.
$$

Every point lands on the diagonal line through the origin, in the direction $(1, 1)$. The column space is that line, so the whole plane gets flattened onto it. Where along the line a point lands depends only on its total, $x_1 + x_2$.

**The bias shifts the line.** Adding $\mathbf{b}^{(1)} = (-0.5, -1.5)$ moves every point by the same amount, so the line moves without tilting: it now passes through $(-0.5, -1.5)$, still in the direction $(1, 1)$. That's the line $z^{(1)}_2 = z^{(1)}_1 - 1$ in the middle panel of row 1.

**Points with the same total land on the same spot.** The class-1 centers $(1, 0)$ and $(0, 1)$ both have total 1, so both land at $1\cdot(1, 1) + (-0.5, -1.5) = (0.5, -0.5)$. The weighted sums can't tell them apart.

(In this example the rows and the columns of $W^{(1)}$ happen to be the same, since the matrix is symmetric. In general, the **rows** are the units' weights, and the **columns** decide where the outputs can land.)

</details>

The rest of this section looks more closely at each layer.

#### Step 1: the hidden layer moves each point

The cliff figure below shows both halves of each hidden unit at once (weighted sum and activation), so its first two small diagrams highlight the hidden units completely. Its third panel looks ahead to the output unit.

Follow one point by hand. Take $\mathbf{x} = (0.9, 0.1)$, which is in the class-1 cluster near $(1, 0)$:

- $z^{(1)}_1 = 0.9 + 0.1 - 0.5 = 0.5 > 0$, so $h_1 = 1$.
- $z^{(1)}_2 = 0.9 + 0.1 - 1.5 = -0.5$, so $h_2 = 0$.

So this point's new coordinates are $(h_1, h_2) = (1, 0)$.

Each hidden unit, on its own, is a single neuron like Section 2's, so it has one line in feature space: the line where it **switches** from 0 to 1 ($z^{(1)}_k = 0$). Hidden unit 1's line is drawn **solid** and hidden unit 2's **dashed**. A point's spot in h-space depends only on which side of each line it's on.

Over feature space, each hidden unit's output is a **cliff** along its own line:

In [ ]:
from ch_6_and_10_demos_helpers import plot_hidden_cliffs

plot_hidden_cliffs();

Panels 1 and 2 are the two hidden units: each is a **cliff** along its own line, 0 on one side and 1 on the other.

Panel 3 looks ahead to the output unit. Its score, $z^{(2)} = h_1 - h_2 - 0.5$, combines the two cliffs: it adds hidden unit 1's cliff, subtracts hidden unit 2's, and lowers the result by 0.5. The score is constant in each of three regions:

- below both lines: $0 - 0 - 0.5 = -0.5$
- between the lines: $1 - 0 - 0.5 = +0.5$
- above both lines: $1 - 1 - 0.5 = -0.5$

A step up followed by a step down makes a **raised plateau** between the lines. Each data point is drawn at its score, and the gray plane marks height 0, where the output unit's activation cuts: points above the plane are predicted class 1. Both blue clusters sit up on the plateau, and both orange clusters sit below the plane. The decision boundary is where the plateau crosses the gray plane: along both hidden lines.

The same three regions explain where the hidden layer sends each point. Every point in a region gets the same $(h_1, h_2)$, so the regions land on three spots in h-space: below both lines goes to $(0, 0)$, between the lines to $(1, 0)$, and above both lines to $(1, 1)$. **Both blue clusters land on the same spot, $(1, 0)$**, even though they sat in opposite corners of feature space (row 1 of the overview strip).

#### Step 2: the output unit draws one line, in h-space

The output unit never sees $x_1$ and $x_2$. Its only inputs are $h_1$ and $h_2$, so it works in h-space (row 2 of the overview strip). There, the three spots sit on opposite sides of one straight line, $z^{(2)} = h_1 - h_2 - 0.5 = 0$, and the output unit's activation cuts at that line.

> **Key idea.** The output unit is an ordinary single neuron, like Section 2's, so its decision boundary is **one straight line**. It just draws that line in **h-space**, not in feature space. The hidden layer's job is to move the points so that a straight line works.

Drawn back in feature space, that one line in h-space becomes the network's decision boundary: **two lines**, the hidden units' switch lines, with class 1 in the band between them.

In [ ]:
from ch_6_and_10_demos_helpers import plot_xor_boundary

plot_xor_boundary();

**Summary of "Spaces":**

| Picture | Space | What it shows |
| --- | --- | --- |
| Demo Q | feature space → new coordinates | a weighted-sum layer only moves the plane; the activation bends it |
| Overview strip, row 1 | feature space → $z^{(1)}$-space → h-space | the hidden layer moves the points |
| Overview strip, row 2 | h-space → $z^{(2)}$-line → out (0 or 1) | the output unit draws one line, then its activation snaps each point to 0 or 1 |
| Cliffs and plateau | feature space | each hidden unit is one cliff; the output score combines them |
| Decision boundary | feature space | the result, on the original data |

(Appendix A walks through the same network with sigmoid hidden units instead of steps.)

<a id="notation"></a>

### Notation: the whole network in four lines

So far we've written each unit separately. To code a network, it's easier to handle a whole layer at once. Here is the network again, with every weight and bias labeled:

In [ ]:
from ch_6_and_10_demos_helpers import plot_network_diagram

plot_network_diagram();

**Each step, written out.** Every unit does the same two things: a weighted sum of its inputs plus its bias, then its activation $g$. Each row below is one circle in the diagram.

| Unit (one circle in the diagram) | Weighted sum | Activation |
| --- | --- | --- |
| **Hidden layer (layer 1)** | | |
| hidden unit 1 | $z^{(1)}_1 = w^{(1)}_{11}x_1 + w^{(1)}_{12}x_2 + b^{(1)}_1$ | $h_1 = g\big(z^{(1)}_1\big)$ |
| hidden unit 2 | $z^{(1)}_2 = w^{(1)}_{21}x_1 + w^{(1)}_{22}x_2 + b^{(1)}_2$ | $h_2 = g\big(z^{(1)}_2\big)$ |
| **Output layer (layer 2)** | | |
| output unit | $z^{(2)} = w^{(2)}_1 h_1 + w^{(2)}_2 h_2 + b^{(2)}$ | $\text{output} = g\big(z^{(2)}\big)$ |

Read across a row for one unit. Read down a column for one kind of step.

**Packing each column into vectors and matrices.** Collect the numbers in the table into boxes. **Bold letters are vectors** (lists of numbers), **plain letters are single numbers**, and **capital letters are matrices**. Vectors are written as columns, as in linear algebra:

$$
\mathbf{x} = \begin{bmatrix} x_1 \\ x_2 \end{bmatrix}, \quad
W^{(1)} = \begin{bmatrix} w^{(1)}_{11} & w^{(1)}_{12} \\ w^{(1)}_{21} & w^{(1)}_{22} \end{bmatrix}, \quad
\mathbf{b}^{(1)} = \begin{bmatrix} b^{(1)}_1 \\ b^{(1)}_2 \end{bmatrix}, \quad
\mathbf{z}^{(1)} = \begin{bmatrix} z^{(1)}_1 \\ z^{(1)}_2 \end{bmatrix}, \quad
\mathbf{h} = \begin{bmatrix} h_1 \\ h_2 \end{bmatrix}, \quad
\mathbf{w}^{(2)} = \begin{bmatrix} w^{(2)}_1 \\ w^{(2)}_2 \end{bmatrix}
$$

**Row $k$ of $W^{(1)}$ holds hidden unit $k$'s weights**, so $w^{(1)}_{kj}$ sits in row $k$, column $j$. The output layer has one unit, so its bias $b^{(2)}$ and score $z^{(2)}$ are single numbers.

With these boxes, each column of the table becomes one line:

$$
\begin{bmatrix} z^{(1)}_1 \\ z^{(1)}_2 \end{bmatrix} =
\begin{bmatrix} w^{(1)}_{11} & w^{(1)}_{12} \\ w^{(1)}_{21} & w^{(1)}_{22} \end{bmatrix}
\begin{bmatrix} x_1 \\ x_2 \end{bmatrix} +
\begin{bmatrix} b^{(1)}_1 \\ b^{(1)}_2 \end{bmatrix},
\qquad
\begin{bmatrix} h_1 \\ h_2 \end{bmatrix} =
\begin{bmatrix} g\big(z^{(1)}_1\big) \\ g\big(z^{(1)}_2\big) \end{bmatrix},
\qquad
z^{(2)} = \mathbf{w}^{(2)}\cdot\mathbf{h} + b^{(2)}
$$

Multiply out the first equation: row $k$ of $W^{(1)}$ dotted with $\mathbf{x}$ gives $w^{(1)}_{k1}x_1 + w^{(1)}_{k2}x_2$, exactly the table's weighted sum for hidden unit $k$. The last equation is a dot product: $\mathbf{w}^{(2)}\cdot\mathbf{h} = w^{(2)}_1 h_1 + w^{(2)}_2 h_2$.

**The forward pass, in matrix form.** Replace each box by its name, and the whole network, from features to prediction, is four lines: a weighted sum and an activation for each layer. These are the four arrows of the overview strip, in order.

$$
\boxed{
\begin{aligned}
\mathbf{z}^{(1)} &= W^{(1)}\mathbf{x} + \mathbf{b}^{(1)} && \text{hidden layer: weighted sums} \\
\mathbf{h} &= g\big(\mathbf{z}^{(1)}\big) && \text{hidden layer: activation, applied to each entry} \\
z^{(2)} &= \mathbf{w}^{(2)}\cdot\mathbf{h} + b^{(2)} && \text{output unit: weighted sum} \\
\text{output} &= g\big(z^{(2)}\big) && \text{output unit: activation}
\end{aligned}
}
$$

**One data point, by hand.** For the XOR network and $\mathbf{x} = (0.9, 0.1)$:

$$
\begin{aligned}
\mathbf{z}^{(1)} &= \begin{bmatrix} 1 & 1 \\ 1 & 1 \end{bmatrix}\begin{bmatrix} 0.9 \\ 0.1 \end{bmatrix} + \begin{bmatrix} -0.5 \\ -1.5 \end{bmatrix} = \begin{bmatrix} 0.5 \\ -0.5 \end{bmatrix} \\
\mathbf{h} &= \text{step}\big(\mathbf{z}^{(1)}\big) = \begin{bmatrix} 1 \\ 0 \end{bmatrix} \\
z^{(2)} &= (1, -1)\cdot(1, 0) - 0.5 = 0.5 \\
\text{output} &= \text{step}(0.5) = 1
\end{aligned}
$$

The data point is predicted class 1, which is correct.

**From one data point to all of them, in code.** In code, forward passes don't act on one point ($x$ vector) at a time. The data sit in a matrix `X` with **one row per data point** and one column per feature, as in every earlier chapter, and one line computes $W^{(1)}\mathbf{x} + \mathbf{b}^{(1)}$ for every row at once:

```python
Z1 = X @ W1.T + b1      # row i holds data point i's two hidden scores
```

**What's the same in the math and in the code, and what's flipped.** The code stores the *weights* exactly as the math does, but it stores the *data* the other way around:

| Object | In the math | In the code | Same or flipped? |
| --- | --- | --- | --- |
| hidden weights | $W^{(1)}$, one row per hidden unit | `W1`, one row per hidden unit | **the same matrix** |
| one data point | $\mathbf{x}$, a column | one **row** of `X` | flipped |
| its hidden scores | $\mathbf{z}^{(1)}$, a column | one **row** of `Z1` | flipped |
| biases | $\mathbf{b}^{(1)}$ | `b1`, a 1-D array (no row or column orientation) | the same numbers |

**Why `X @ W1.T + b1`.** Start from the math, $\mathbf{z}^{(1)} = W^{(1)}\mathbf{x} + \mathbf{b}^{(1)}$, and transpose both sides using one rule of matrix algebra: **transposing a product reverses its order**, $(AB)^\top = B^\top A^\top$.

$$
\mathbf{z}^{(1)\top} = \mathbf{x}^\top\, W^{(1)\top} + \mathbf{b}^{(1)\top}
$$

Now match each piece to the code:

- $\mathbf{x}^\top$ is the data point written as a **row**. That's exactly how the code stores a data point (one row of `X`), so in code it's just `x`, with no transpose.
- $W^{(1)\top}$ is the transpose of $W^{(1)}$, and the code stores $W^{(1)}$ as `W1`. So this transpose stays: `W1.T`.
- $\mathbf{z}^{(1)\top}$ is the scores written as a row: one row of `Z1`.

So $\mathbf{x}^\top W^{(1)\top} + \mathbf{b}^{(1)\top}$ is `x @ W1.T + b1`, and with every data point as a row of `X`, it's `X @ W1.T + b1`. These are the same numbers as $W^{(1)}\mathbf{x} + \mathbf{b}^{(1)}$, written as rows, so everything this notebook reasoned with $W\mathbf{x}$ applies to the code unchanged. The code check after the numpy cell below confirms it with numbers.

**A worked example.** Our XOR matrix is symmetric, so its transpose looks identical, which hides the flip. This example uses a different hidden layer instead: unit 1's weights are $(1, 2)$ and unit 2's are $(-1, 1)$. The three rows of $X$ are three data points.

In [ ]:
from ch_6_and_10_demos_helpers import plot_transpose_example

plot_transpose_example();

The highlighted row of $X$ (data point 3, which is $(2, 1)$) meets the highlighted column of $W^{(1)\top}$ (unit 1's weights, $(1, 2)$): $2\cdot 1 + 1\cdot 2 = 4$, which is data point 3's score for hidden unit 1. As a check, the one-point formula gives the same pair of scores:

$$
W^{(1)}\begin{bmatrix} 2 \\ 1 \end{bmatrix} = \begin{bmatrix} 1 & 2 \\ -1 & 1 \end{bmatrix}\begin{bmatrix} 2 \\ 1 \end{bmatrix} = \begin{bmatrix} 4 \\ -1 \end{bmatrix},
$$

the same two numbers as row 3 of $Z^{(1)}$. Without the transpose, `X @ W1` would pair each data point with a *column* of $W^{(1)}$, which mixes the two units' weights together. Its row 3 would be $(1, 5)$, which isn't either unit's score.

Adding `b1` adds each unit's bias to every row. The output layer needs no transpose: `w2` is a plain list of 2 numbers, so `H @ w2` takes the dot product of each row of `H` with `w2`.

Each line of the code does one line of the boxed forward pass, for every data point at once:

| Math (one data point) | Code (all data points at once) | Shape of the result |
| --- | --- | --- |
| $\mathbf{z}^{(1)} = W^{(1)}\mathbf{x} + \mathbf{b}^{(1)}$ | `Z1 = X_xor @ W1.T + b1` | (100, 2): one row per data point |
| $\mathbf{h} = \text{step}\big(\mathbf{z}^{(1)}\big)$ | `H = (Z1 > 0).astype(float)` | (100, 2) |
| $z^{(2)} = \mathbf{w}^{(2)}\cdot\mathbf{h} + b^{(2)}$ | `z2 = H @ w2 + b2` | (100,): one score per data point |
| $\text{output} = \text{step}\big(z^{(2)}\big)$ | `pred = (z2 > 0).astype(int)` | (100,) |

**Code worth reading.** The same network in numpy, checked on all 100 points.

In [ ]:
# Layer 1 (hidden): one row per hidden unit
W1 = np.array([[1.0, 1.0],     # unit 1's weights: z = x1 + x2 - 0.5 ("at least one is on")
               [1.0, 1.0]])    # unit 2's weights: z = x1 + x2 - 1.5 ("both are on")
b1 = np.array([-0.5, -1.5])    # biases: (unit 1, unit 2)

# Layer 2 (output): one unit
w2 = np.array([1.0, -1.0])     # weights on h1 and h2: z = h1 - h2 - 0.5
b2 = -0.5

Z1 = X_xor @ W1.T + b1               # layer 1's scores: one row per data point, one column per unit
H = (Z1 > 0).astype(float)           # h1, h2: the step activation
z2 = H @ w2 + b2                     # layer 2's score
pred = (z2 > 0).astype(int)

print('h1, h2, and the output score for the first three data points:')
print(np.column_stack([H[:3], z2[:3]]))
print('mistakes:', np.sum(pred != y_xor))

**Code worth reading.** A check that the two forms agree. Take one data point, compute $W^{(1)}\mathbf{x} + \mathbf{b}^{(1)}$ the math way, and compare it with that data point's row of `X_xor @ W1.T + b1`. (numpy stores one data point as a plain 1-D array, so `W1 @ x` treats it as a column.)

In [ ]:
x = X_xor[2]                              # one data point
math_way = W1 @ x + b1                    # z = W x + b, as in the math
code_way = (X_xor @ W1.T + b1)[2]         # row 2 of the all-points version
print('W x + b:          ', math_way)
print('row of X @ W.T + b:', code_way)
print('same numbers?', np.allclose(math_way, code_way))

`X_xor @ W1.T + b1` computes both hidden units' scores for all 100 data points at once: a 100 × 2 array, one row per data point and one column per unit. Count the numbers we chose: 4 weights and 2 biases in the hidden layer, plus 2 weights and 1 bias in the output unit, **9 parameters** in all.

<a id="activation-matters"></a>

### Why the activation can't be left out

Everything above relied on the hidden units' surfaces being *bent*: cliffs, or S-shaped slopes. Suppose instead that the hidden units had no activation, so $h_1 = z^{(1)}_1$ and $h_2 = z^{(1)}_2$. Then each hidden surface is a tilted plane, and the output score adds tilted planes:

$$
\begin{aligned}
z^{(2)} &= w^{(2)}_1\big(w^{(1)}_{11}x_1 + w^{(1)}_{12}x_2 + b^{(1)}_1\big) + w^{(2)}_2\big(w^{(1)}_{21}x_1 + w^{(1)}_{22}x_2 + b^{(1)}_2\big) + b^{(2)} \\
&= \underbrace{\big(w^{(2)}_1 w^{(1)}_{11} + w^{(2)}_2 w^{(1)}_{21}\big)}_{\text{one weight}}\,x_1 + \underbrace{\big(w^{(2)}_1 w^{(1)}_{12} + w^{(2)}_2 w^{(1)}_{22}\big)}_{\text{one weight}}\,x_2 + \underbrace{\big(w^{(2)}_1 b^{(1)}_1 + w^{(2)}_2 b^{(1)}_2 + b^{(2)}\big)}_{\text{one bias}}
\end{aligned}
$$

That's two weights and one bias: a single neuron again. A sum of tilted planes is a tilted plane, and its zero level is one line. **The activation is what gives each hidden unit's surface a bend, and adding bent surfaces is what lets the boundary bend.**

# Practice Questions

These questions follow the notebook in order: one neuron, its boundary, its limits, then networks. Questions like these will appear on the learning check, so try each one on paper (calculator is ok) before you open its hint or answer.

For reference:

- One neuron: score $z = \mathbf{w}\cdot\mathbf{x} + b = w_1x_1 + w_2x_2 + b$, output $g(z)$.
- Activations: sigmoid $g(z) = \dfrac{1}{1 + e^{-z}}$; step: 1 if $z > 0$, else 0; ReLU: $\max(0, z)$.
- $w^{(\ell)}_{kj}$: in layer $\ell$, the weight into unit $k$ from input $j$.
- Forward pass, one data point $\mathbf{x}$: $\mathbf{z}^{(1)} = W^{(1)}\mathbf{x} + \mathbf{b}^{(1)}$, $\;\mathbf{h} = g\big(\mathbf{z}^{(1)}\big)$, $\;z^{(2)} = \mathbf{w}^{(2)}\cdot\mathbf{h} + b^{(2)}$, $\;\text{output} = g\big(z^{(2)}\big)$. Row $k$ of $W^{(1)}$ holds hidden unit $k$'s weights.
- All data points at once, in code: `X @ W1.T + b1`, with one row of `X` per data point.

### Question 1: Vocabulary, one neuron

Match each description to its term.

| Description | | Term |
| --- | --- | --- |
| i. The number a neuron multiplies one of its inputs by. | | A. neuron, or unit |
| ii. The function applied to the score, such as the sigmoid, the step, or ReLU. | | B. weight |
| iii. A weighted sum of inputs plus a bias, followed by an activation function. | | C. bias |
| iv. The constant added to the weighted sum; it shifts where the neuron switches. | | D. score, $z$ |
| v. A neuron that uses the step activation function. | | E. activation function, $g$ |
| vi. The value a unit outputs: the result of its activation function applied to its score. | | F. activation, $h = g(z)$ |
| vii. The weighted sum of a neuron's inputs plus its bias, before the activation function is applied. | | G. perceptron |

(See [Section 1](#section-1) and [Section 4](#section-4).)

<details>
<summary>Answer</summary>

i. B, ii. E, iii. A, iv. C, v. G, vi. F, vii. D.

Watch the difference between E and F: the activation **function** $g$ is the rule, such as the sigmoid. The **activation** $h$ is the number it produces for one data point.

</details>

### Question 2: Vocabulary, networks and their pictures

Match each description to its term.

| Description | | Term |
| --- | --- | --- |
| i. The plane whose axes are the features; each data point is a dot. | | A. layer |
| ii. Where the model's prediction switches from one class to the other. | | B. hidden unit |
| iii. A unit that is neither an input nor the output: its activation feeds other units. | | C. output unit |
| iv. All the weights and biases: the numbers that define a network. | | D. feature space |
| v. A group of units that all read the same inputs. | | E. weight space |
| vi. The space whose axes are a model's parameters (its weights and biases); each point is one whole model. Pictures usually show just two of the axes. | | F. h-space (a hidden representation, or latent space) |
| vii. Computing a network's output from its input, one layer after another. | | G. decision boundary |
| viii. The unit whose activation is the network's prediction. | | H. forward pass |
| ix. The plane whose axes are the hidden units' activations. | | I. parameters |

(See [Section 2](#section-2) and [Section 4](#section-4).)

<details>
<summary>Answer</summary>

i. D, ii. G, iii. B, iv. I, v. A, vi. E, vii. H, viii. C, ix. F.

</details>

### Question 3: Name the symbol

For the one-hidden-layer network in the [notation section](#notation), say what each symbol is called and where it appears in the network diagram: on an **input**, on an **edge**, **inside a unit**, or **leaving a unit**.

$x_j$, $\quad w^{(1)}_{kj}$, $\quad b^{(1)}_k$, $\quad b^{(2)}$, $\quad z^{(1)}_k$, $\quad g$, $\quad h_k$, $\quad w^{(2)}_k$, $\quad z^{(2)}$

Then: what are $W^{(1)}$ and $X$?

<details>
<summary>Answer</summary>

| Symbol | Name | In the diagram |
| --- | --- | --- |
| $x_j$ | feature $j$ of a data point | an input |
| $w^{(1)}_{kj}$ | the weight into hidden unit $k$ from input $j$ | the edge from $x_j$ to hidden unit $k$ |
| $b^{(1)}_k$ | hidden unit $k$'s bias (layer 1) | the edge from the "1" node in the input column into hidden unit $k$ |
| $b^{(2)}$ | the output unit's bias (layer 2) | the edge from the "1" node below the hidden layer into the output unit |
| $z^{(1)}_k$ | hidden unit $k$'s score (its weighted sum plus bias)  | inside hidden unit $k$ |
| $g$ | the activation function | inside every unit, applied to its score |
| $h_k$ | hidden unit $k$'s activation (its output) | leaving hidden unit $k$ |
| $w^{(2)}_k$ | the weight from $h_k$ into the output unit | the edge from hidden unit $k$ to the output unit |
| $z^{(2)}$ | the output unit's score | inside the output unit |

$W^{(1)}$ is the matrix of all the hidden layer's weights: row $k$ holds hidden unit $k$'s weights, and column $j$ holds the weights on input $j$. $X$ is the data matrix: one row per data point, one column per feature.

</details>

### Question 4: Logistic regression as a neuron

A logistic regression has log-odds $1.5 - 0.8X_1 + 2X_2$. (a) Write it as a neuron: what are $\mathbf{w}$, $b$, and $g$? (b) What does it output for $\mathbf{x} = (2, 1)$? (See [Section 1](#section-1).)

<details>
<summary>Answer</summary>

(a) $\mathbf{w} = (-0.8, 2)$, $b = 1.5$, and $g$ is the sigmoid.

(b) $z = -0.8(2) + 2(1) + 1.5 = 1.9$, so the output is $\dfrac{1}{1 + e^{-1.9}} \approx 0.87$.

</details>

### Question 5: Three activation functions

Compute the sigmoid, the step, and ReLU at $z = -2$, $z = 0$, and $z = 1.5$. Which of the three gives a probability? (See [Section 1](#section-1).)

<details>
<summary>Hint</summary>

$e^{2} \approx 7.39$ and $e^{-1.5} \approx 0.22$.

</details>

<details>
<summary>Answer</summary>

| $z$ | sigmoid | step | ReLU |
| --- | --- | --- | --- |
| $-2$ | $\approx 0.12$ | 0 | 0 |
| $0$ | $0.5$ | 0 | 0 |
| $1.5$ | $\approx 0.82$ | 1 | 1.5 |

Only the sigmoid gives a probability. The step at $z = 0$ gives 0, because it outputs 1 only when $z > 0$.

</details>

### Question 6: One neuron, by hand

A neuron has $\mathbf{w} = (1, -2)$ and $b = 1$. For each data point, $(3, 1)$ and $(1, 2)$: (a) compute the score $z$; (b) give the output with a step, with ReLU, and with a sigmoid. (See [Section 1](#section-1).)

<details>
<summary>Answer</summary>

**$(3, 1)$:** $z = 1(3) - 2(1) + 1 = 2$. Step: 1. ReLU: 2. Sigmoid: $\dfrac{1}{1 + e^{-2}} \approx 0.88$.

**$(1, 2)$:** $z = 1(1) - 2(2) + 1 = -2$. Step: 0. ReLU: 0. Sigmoid: $\dfrac{1}{1 + e^{2}} \approx 0.12$.

</details>

### Question 7: Which plane?

Each description below is of a picture. Is the picture drawn in **feature space** or **weight space**?

(a) The axes are $x_1$ and $x_2$, and each dot is one customer.
(b) Each dot is a whole model, and the RSS ovals are drawn around the least squares fit.
(c) A model shows up as a line that splits the plane in two.
(d) Gradient descent's path walks across the picture, one step per update. (See [Section 2](#section-2).)

<details>
<summary>Answer</summary>

(a) Feature space. (b) Weight space (ridge's $\beta_1, \beta_2$ pictures). (c) Feature space: the line is the model's decision boundary. (d) Weight space: gradient descent changes the weights, so each step moves to a new model.

</details>

### Question 8: Could gradient descent train a step neuron?

Explain, in two or three sentences, why a network of step neurons can't be trained by gradient descent. (See [Section 2](#section-2).)

<details>
<summary>Hint</summary>

Take a data point with score $z = 0.8$. Nudge one weight so that $z = 0.81$. What happens to the step's output, and so to the loss?

</details>

<details>
<summary>Answer</summary>

Nudging a weight changes a step neuron's score $z$ a little, but unless $z$ crosses 0 its output doesn't change at all, so the loss doesn't change either. The slope of the loss is 0 almost everywhere, so gradient descent's update (old value $- \eta \times$ slope) never moves. Activations like the sigmoid and ReLU have slopes that carry information.

</details>

### Question 9: AND, OR, XOR

Data points at the four corners $(0, 0)$, $(0, 1)$, $(1, 0)$, $(1, 1)$ are labeled three ways. AND: only $(1, 1)$ is class 1. OR: every corner except $(0, 0)$ is class 1. XOR: $(0, 1)$ and $(1, 0)$ are class 1. For each, can one step neuron classify all four corners correctly? If so, give a $\mathbf{w}$ and $b$ that work. (See [Section 3](#section-3).)

<details>
<summary>Hint</summary>

Try $\mathbf{w} = (1, 1)$, so the score is $x_1 + x_2 + b$. What is $x_1 + x_2$ at each corner? Where would you put the cutoff?

</details>

<details>
<summary>Answer</summary>

AND and OR can; XOR can't.

- **AND:** $\mathbf{w} = (1, 1)$, $b = -1.5$. The score $x_1 + x_2 - 1.5$ is positive only at $(1, 1)$.
- **OR:** $\mathbf{w} = (1, 1)$, $b = -0.5$. The score is positive everywhere except $(0, 0)$.
- **XOR:** no line works. The segment between the two class-1 corners crosses the segment between the two class-0 corners, and a single line can't put that crossing point on both sides.

</details>

### Question 10: Reading a network

A network shaped like the one in the [network diagram](#notation) has these weights and biases:

| | from $x_1$ | from $x_2$ | bias |
| --- | --- | --- | --- |
| into hidden unit 1 | $w^{(1)}_{11} = 2$ | $w^{(1)}_{12} = -1$ | $b^{(1)}_1 = 0$ |
| into hidden unit 2 | $w^{(1)}_{21} = 1$ | $w^{(1)}_{22} = 1$ | $b^{(1)}_2 = -2$ |

and output weights $w^{(2)}_1 = 1$, $w^{(2)}_2 = 1$, $b^{(2)} = -1$.

(a) Which edge of the diagram carries $w^{(1)}_{21}$? (b) Write out $z^{(1)}_2$ in terms of $x_1$ and $x_2$. (c) How many parameters does the network have? (See [Section 4](#section-4).)

<details>
<summary>Answer</summary>

(a) The edge **into hidden unit 2, from $x_1$**: the first subscript is the unit, the second is the input. Its value is 1.

(b) $z^{(1)}_2 = w^{(1)}_{21}x_1 + w^{(1)}_{22}x_2 + b^{(1)}_2 = x_1 + x_2 - 2$.

(c) $4 + 2 = 6$ in the hidden layer (4 weights, 2 biases), plus $2 + 1 = 3$ in the output layer: **9**.

</details>

### Question 11: What can a layer of weighted sums do?

True or false: a layer of weighted sums (no activation) can (a) shift the plane; (b) turn and stretch it; (c) flatten the whole plane onto a line; (d) bend straight grid lines into curves; (e) make two classes separable by a line when no line separated them before. Then explain why weighted sums alone can't make the XOR data separable. (See [Demo Q](#demo-q).)

<details>
<summary>Answer</summary>

(a) True. (b) True. (c) True: our XOR hidden layer does exactly that. (d) False. (e) False.

A layer of weighted sums keeps straight lines straight and parallel lines parallel. For XOR, it can even send $(1, 0)$ and $(0, 1)$ to the same spot (flattening the plane onto a line), but the two class-0 clusters land on either side of that spot, so the classes are still interleaved: orange, blue, orange along the line. Only an activation can bend or snap the plane so that one line works.

</details>

### Question 12: Follow one data point through the XOR network

The XOR network has $W^{(1)} = \begin{bmatrix} 1 & 1 \\ 1 & 1 \end{bmatrix}$, $\mathbf{b}^{(1)} = \begin{bmatrix} -0.5 \\ -1.5 \end{bmatrix}$, $\mathbf{w}^{(2)} = (1, -1)$, $b^{(2)} = -0.5$, and step activations. Push the data point $\mathbf{x} = (0.1, 0.9)$ through it: (a) $\mathbf{z}^{(1)}$; (b) $\mathbf{h}$; (c) $z^{(2)}$ and the prediction. (d) Which spot in h-space does it land on, and which other cluster of the XOR data lands on the same spot? (See [Following the data through the network](#forward-pass).)

<details>
<summary>Answer</summary>

(a) $\mathbf{z}^{(1)} = (0.1 + 0.9 - 0.5,\; 0.1 + 0.9 - 1.5) = (0.5, -0.5)$.

(b) $\mathbf{h} = (\text{step}(0.5), \text{step}(-0.5)) = (1, 0)$.

(c) $z^{(2)} = 1(1) + (-1)(0) - 0.5 = 0.5 > 0$, so the prediction is **class 1**. Correct: $(0.1, 0.9)$ is near the class-1 corner $(0, 1)$.

(d) It lands at $(1, 0)$ in h-space, the same spot as the cluster near $(1, 0)$ in feature space, the other class-1 cluster.

</details>

### Question 13: One line, or two?

In h-space, the XOR network's output unit draws **one** straight line. In feature space, its decision boundary is **two** lines. How can both be true? (See [Step 2](#forward-pass).)

<details>
<summary>Hint</summary>

Where does each region of feature space land in h-space? Which side of the output unit's line is each spot on?

</details>

<details>
<summary>Answer</summary>

The output unit only sees $(h_1, h_2)$, and in that space its boundary $z^{(2)} = 0$ is one line. But the hidden layer has already moved the data points: each of the three regions of feature space (below, between, and above the two hidden lines) lands on a single spot of h-space. The one line in h-space separates the spot for the middle region from the other two. Back in feature space, "the middle region" is the band between the two hidden lines, so its edges, the two hidden lines, form the boundary. On the plateau in panel 3, those are the two places where the surface crosses the gray plane.

</details>

### Question 14: A forward pass by hand

A small network has a hidden layer with **ReLU** activations and an output unit with a **step**:

$$
W^{(1)} = \begin{bmatrix} 1 & -1 \\ 0 & 2 \end{bmatrix}, \quad
\mathbf{b}^{(1)} = \begin{bmatrix} 0 \\ -1 \end{bmatrix}, \quad
\mathbf{w}^{(2)} = (1, -1), \quad
b^{(2)} = 0.5
$$

(a) For the data point $\mathbf{x} = (1, 2)$, compute $z^{(1)}_1$ and $z^{(1)}_2$ one unit at a time, using $z^{(1)}_k = w^{(1)}_{k1}x_1 + w^{(1)}_{k2}x_2 + b^{(1)}_k$.
(b) Compute $\mathbf{h}$, then $z^{(2)}$, then the prediction.
(c) Check (a) with the matrix form $W^{(1)}\mathbf{x} + \mathbf{b}^{(1)}$.
(d) Do the same for the data point $(3, 0)$. Then: in numpy, with `X = np.array([[1, 2], [3, 0]])`, what does `X @ W1.T + b1` return? (See [the notation section](#notation).)

<details>
<summary>Hint</summary>

Row $k$ of $W^{(1)}$ holds hidden unit $k$'s weights. So hidden unit 1's weights are $(1, -1)$ and hidden unit 2's are $(0, 2)$.

</details>

<details>
<summary>Answer</summary>

(a) $z^{(1)}_1 = 1(1) + (-1)(2) + 0 = -1$, and $z^{(1)}_2 = 0(1) + 2(2) - 1 = 3$.

(b) $\mathbf{h} = (\max(0, -1), \max(0, 3)) = (0, 3)$. Then $z^{(2)} = 1(0) + (-1)(3) + 0.5 = -2.5$, so the prediction is **0**.

(c) $\begin{bmatrix} 1 & -1 \\ 0 & 2 \end{bmatrix}\begin{bmatrix} 1 \\ 2 \end{bmatrix} + \begin{bmatrix} 0 \\ -1 \end{bmatrix} = \begin{bmatrix} 1 - 2 \\ 0 + 4 \end{bmatrix} + \begin{bmatrix} 0 \\ -1 \end{bmatrix} = \begin{bmatrix} -1 \\ 3 \end{bmatrix}$, as in (a).

(d) For $(3, 0)$: $\mathbf{z}^{(1)} = (3, -1)$, $\mathbf{h} = (3, 0)$, $z^{(2)} = 3 - 0 + 0.5 = 3.5$, so the prediction is **1**. `X @ W1.T + b1` returns both data points' hidden scores at once, one row per data point: `[[-1, 3], [3, -1]]`.

</details>

### Question 15: Shapes and parameters

A network has **3 features**, a hidden layer of **4 units**, and **1 output unit**, and the data have $n = 50$ data points. (a) Give the shape of $W^{(1)}$, $\mathbf{b}^{(1)}$, and $\mathbf{w}^{(2)}$. (b) In numpy, give the shape of `X`, of `Z1 = X @ W1.T + b1`, and of `z2 = H @ w2 + b2`. (c) How many parameters does the network have? (See [the notation section](#notation).)

<details>
<summary>Hint</summary>

$W^{(1)}$ has one row per hidden unit and one column per input. In `X @ W1.T`, the inner sizes must match: $(50 \times 3)(3 \times ?)$.

</details>

<details>
<summary>Answer</summary>

(a) $W^{(1)}$: $4 \times 3$ (one row per hidden unit, one column per input). $\mathbf{b}^{(1)}$: 4 numbers. $\mathbf{w}^{(2)}$: 4 numbers.

(b) `X`: (50, 3). `Z1`: (50, 4), one row per data point and one column per hidden unit. `z2`: (50,), one score per data point.

(c) $4 \times 3 + 4 = 16$ in the hidden layer, plus $4 + 1 = 5$ in the output layer: **21**.

</details>

### Question 16: Same two lines, different shapes

Two hidden units with step activations draw crossing lines: $h_1 = 1$ when $x_1 > 0.5$ (right of the solid line), and $h_2 = 1$ when $x_2 > 0.5$ (above the dashed line). The lines cut feature space into four regions, A to D:

In [ ]:
from ch_6_and_10_demos_helpers import plot_two_lines_blank

plot_two_lines_blank();

(a) What is $(h_1, h_2)$ in each region?
(b) The output unit's score is $z^{(2)} = h_1 + h_2 - 0.5$. What is $z^{(2)}$ in each region, and which regions are predicted class 1 (blue)? Describe the blue shape in words.
(c) Repeat (b) for $z^{(2)} = h_1 + h_2 - 1.5$ and for $z^{(2)} = h_1 - h_2 - 0.5$.
(d) Can **any** output unit, $z^{(2)} = w^{(2)}_1 h_1 + w^{(2)}_2 h_2 + b^{(2)}$, make the XOR pattern: blue in regions B and C only?

<details>
<summary>Hint for (d)</summary>

Write one condition per region. For example, region A is orange, so $w^{(2)}_1 \cdot 0 + w^{(2)}_2 \cdot 0 + b^{(2)} < 0$. Then try adding two of the conditions together.

</details>

<details>
<summary>Answer</summary>

(a) A: $(0, 0)$. B: $(1, 0)$. C: $(0, 1)$. D: $(1, 1)$.

(b) $z^{(2)}$ is $-0.5$ in A, $+0.5$ in B and C, and $+1.5$ in D. Blue: B, C, and D, an **L-shaped** region, "right of the solid line *or* above the dashed line."

(c) For $h_1 + h_2 - 1.5$: only D is positive ($+0.5$), so the blue region is the **top-right corner**, "right of the solid line *and* above the dashed line." For $h_1 - h_2 - 0.5$: only B is positive ($+0.5$), the **bottom-right corner**, "right of the solid line but not above the dashed one."

(d) No. The output unit would need

- A orange: $b^{(2)} < 0$
- B blue: $w^{(2)}_1 + b^{(2)} > 0$
- C blue: $w^{(2)}_2 + b^{(2)} > 0$
- D orange: $w^{(2)}_1 + w^{(2)}_2 + b^{(2)} < 0$

Adding the two middle conditions gives $w^{(2)}_1 + w^{(2)}_2 + 2b^{(2)} > 0$, so $w^{(2)}_1 + w^{(2)}_2 + b^{(2)} > -b^{(2)}$, and $-b^{(2)}$ is positive because $b^{(2)} < 0$. That contradicts the condition for D.

In the $(h_1, h_2)$ plane, this is XOR again: the four regions land at the corners of a square, with the blue ones diagonally opposite. That's why the XOR network placed its two hidden lines *parallel*, along the diagonal: then both class-1 clusters land on the same spot, and one line can select it. **Where the hidden units put their lines matters as much as how the output unit combines them.**

To see all of these, run `plot_two_line_combinations()` from the helpers.

</details>

### Question 17: Collapsing layers

A network has **no activations** (each unit just outputs its score). Its weights are

$$
W^{(1)} = \begin{bmatrix} 1 & 2 \\ 0 & 1 \end{bmatrix}, \quad
\mathbf{b}^{(1)} = \begin{bmatrix} 1 \\ 0 \end{bmatrix}, \quad
\mathbf{w}^{(2)} = (3, -1), \quad
b^{(2)} = 2.
$$

(a) Find the single neuron, $z = w_1x_1 + w_2x_2 + b$, that this whole network equals. (b) Why does this mean the network can't classify XOR? (See [Why the activation can't be left out](#activation-matters).)

<details>
<summary>Hint</summary>

Write out $z^{(1)}_1$ and $z^{(1)}_2$ (row $k$ of $W^{(1)}$ is unit $k$'s weights), substitute them for $h_1$ and $h_2$ in $z^{(2)}$, and collect the $x_1$ terms, the $x_2$ terms, and the constants.

</details>

<details>
<summary>Answer</summary>

(a) The hidden scores are $z^{(1)}_1 = x_1 + 2x_2 + 1$ and $z^{(1)}_2 = x_2$. With no activation, $h_k = z^{(1)}_k$, so

$$
z^{(2)} = 3(x_1 + 2x_2 + 1) - x_2 + 2 = 3x_1 + 5x_2 + 5.
$$

So $w_1 = 3$, $w_2 = 5$, $b = 5$. (In matrix form: the new weights are $W^{(1)\top}\mathbf{w}^{(2)} = (3, 5)$, and the new bias is $\mathbf{w}^{(2)}\cdot\mathbf{b}^{(1)} + b^{(2)} = 5$.)

(b) It's one neuron, so its boundary is one line, and no line separates XOR.

</details>

## Optional: a neuron's boundary line in vector form

This question uses the vector picture of a line from the linear algebra refresher. It isn't needed for the rest of the notebook.

### Optional Question: Where is the line, and which way does $\mathbf{w}$ point?

A step neuron has $\mathbf{w} = (2, -1)$ and $b = -1$. (a) Write its decision boundary as an equation, and solve it for $x_2$. (b) Which side of the line is predicted class 1, and which way does $\mathbf{w}$ point? Check that $\mathbf{w}$ is perpendicular to the line. (c) If $\mathbf{w}$ and $b$ are both doubled, does the line move? What changes if the neuron uses a sigmoid?

<details>
<summary>Hint: lines in vector form</summary>

The decision boundary is the set of data points where the score is zero: $\mathbf{w}\cdot\mathbf{x} + b = 0$.

- **Why that's a line.** The dot product $\mathbf{w}\cdot\mathbf{x}$ equals $\lVert\mathbf{w}\rVert$ times the signed length of $\mathbf{x}$'s shadow on the direction of $\mathbf{w}$. Every data point with the same shadow length has the same score, and those points form a line **perpendicular to $\mathbf{w}$**. The boundary is the one with score exactly 0.
- **Which side is which.** Moving in the direction of $\mathbf{w}$ makes the shadow longer, so the score goes up. So $\mathbf{w}$ points toward the side where $\mathbf{w}\cdot\mathbf{x} + b > 0$.
- **Checking perpendicular.** Two vectors are perpendicular when their dot product is 0. Find a vector that runs *along* the line (the difference between two points on it), and dot it with $\mathbf{w}$.

The linear algebra refresher's Section 4 has a demo of all three.

</details>

<details>
<summary>Answer</summary>

(a) $2x_1 - x_2 - 1 = 0$, so $x_2 = 2x_1 - 1$.

(b) Class 1 is where $2x_1 - x_2 - 1 > 0$, which is $x_2 < 2x_1 - 1$: the side **below and to the right** of the line. $\mathbf{w} = (2, -1)$ points right and down, toward that side. Two points on the line are $(0, -1)$ and $(1, 1)$, so $(1, 2)$ runs along it, and $(2, -1)\cdot(1, 2) = 2 - 2 = 0$: perpendicular.

(c) The line doesn't move: $2(\mathbf{w}\cdot\mathbf{x} + b) = 0$ exactly where $\mathbf{w}\cdot\mathbf{x} + b = 0$. A step neuron's predictions don't change either, since every score keeps its sign. A sigmoid's probabilities get **sharper**: those above 0.5 move toward 1, and those below move toward 0.

</details>

<a id="appendix-a"></a>

## Appendix A · The same network with sigmoid hidden units

Section 4 used step hidden units so the arithmetic was exact. Here is the same XOR network with the sigmoid instead. Each hidden unit's weights and bias are multiplied by 6, so $z^{(1)}_1 = 6(x_1 + x_2 - 0.5)$ and $z^{(1)}_2 = 6(x_1 + x_2 - 1.5)$: the same lines as before, with sharper slopes. The output unit is unchanged, $z^{(2)} = h_1 - h_2 - 0.5$.

**Step 1: the hidden units.** The cliffs become S-shaped slopes.

In [ ]:
from ch_6_and_10_demos_helpers import plot_hidden_cliffs, plot_hspace, plot_number_line, plot_plateau, plot_xor_boundary

plot_hidden_cliffs(activation='sigmoid');

**Step 2: the output unit in h-space.** The points no longer pile onto three spots: $h_1$ and $h_2$ take values between 0 and 1, so each cluster spreads out along a curve. But the output unit still faces the same simple problem: one straight line in h-space separates the classes.

The left panel draws the output unit's score formula, $z^{(2)} = h_1 - h_2 - 0.5$, over all of h-space. Because the formula is a weighted sum plus a bias, it's a tilted plane, like Section 2's first panel. The data points lie on that plane at their scores: blue above height 0, orange below.

In [ ]:
plot_hspace(activation='sigmoid');

**Step 3: the number line.** The scores now spread out instead of sitting at exactly $\pm 0.5$, but the cut at 0 still sorts them correctly.

In [ ]:
plot_number_line(activation='sigmoid');

**Back in feature space.** The plateau becomes a smooth ridge, and the gray plane at height 0 still separates the classes.

In [ ]:
plot_plateau(activation='sigmoid');

The boundary is still two lines here, because both hidden units point in the same direction, $(1, 1)$, so every surface changes only along that direction. With hidden units pointing in *different* directions, the smooth activations round the corners, and the boundary bends into curves.

In [ ]:
plot_xor_boundary(activation='sigmoid');